# 04 — Shared Data Pipeline Smoke Test (no training)
Checks `src/dataset.py` and `src/transforms.py` on the real data before any model is trained. Every check must print **PASS**; an assertion error means stop and fix.

The test split is read only by the full-decode integrity check. No test batch is built, and no model is involved.
A CPU runtime is enough.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json

from src.utils import load_config, get_paths, set_seed, environment_info

cfg = load_config(REPO_DIR / "configs" / "config.yaml")
paths = get_paths(cfg)
set_seed(cfg["project"]["seed"])
print(json.dumps(environment_info(), indent=2))
print("dataset_root:", paths.dataset_root)

In [ ]:
from src.dataset import load_frozen_split

split_df = load_frozen_split(cfg)  # SHA256 + counts + mapping + every file exists
counts = split_df["split"].value_counts().to_dict()
assert counts == cfg["split"]["expected_counts"]
assert split_df["class_index"].between(0, cfg["split"]["num_classes"] - 1).all()
assert (split_df.groupby("class_name")["class_index"].nunique() == 1).all()
print("PASS frozen split: both SHA256 verified,", counts, "| all", len(split_df), "image files exist")
print(split_df.groupby(["class_index", "class_name"]).size().rename("images").reset_index().to_string(index=False))

In [ ]:
from src.dataset import verify_images_decode

failures = verify_images_decode(split_df, expected_size=(524, 524))
assert failures.empty, failures.head()
print(f"PASS full decode: all {len(split_df)} images decode as RGB 524x524")

In [ ]:
import time

import torch

from src.dataset import get_dataloaders

train_loader, val_loader = get_dataloaders(cfg, split_df=split_df)
size = cfg["project"]["image_size"]
by_filepath = split_df.set_index("filepath")

for name, loader in (("train", train_loader), ("val", val_loader)):
    images, labels, filepaths = next(iter(loader))
    rows = by_filepath.loc[list(filepaths)]
    assert images.shape == (cfg["training"]["batch_size"], 3, size, size), images.shape
    assert images.dtype == torch.float32 and labels.dtype == torch.int64
    assert (rows["split"] == name).all(), "batch contains rows from another split"
    assert (rows["class_index"].to_numpy() == labels.numpy()).all(), "labels do not match class_mapping"
    print(f"PASS {name} batch: {tuple(images.shape)}, labels {labels.min().item()}..{labels.max().item()}, "
          f"pixel mean {images.mean():.3f}, std {images.std():.3f}")
print("batches per epoch: train", len(train_loader), "| val", len(val_loader))

N_BATCHES = 20
start = time.time()
for i, _ in enumerate(train_loader):
    if i + 1 == N_BATCHES:
        break
seconds = time.time() - start
print(f"{N_BATCHES} train batches in {seconds:.1f}s -> about "
      f"{seconds / N_BATCHES * len(train_loader) / 60:.1f} min of data loading per epoch")
print("If that is several minutes, copy the dataset to /content and set paths.dataset_root_override.")

In [ ]:
val_set, train_set = val_loader.dataset, train_loader.dataset
assert torch.equal(val_set[0][0], val_set[0][0]), "validation transform is not deterministic"
assert not torch.equal(train_set[0][0], train_set[0][0]), "training augmentation is not active"
print("PASS validation transform deterministic; training augmentation active")

In [ ]:
import matplotlib.pyplot as plt

mean = torch.tensor(cfg["data"]["normalization"]["mean"]).view(3, 1, 1)
std = torch.tensor(cfg["data"]["normalization"]["std"]).view(3, 1, 1)
class_names = split_df.drop_duplicates("class_index").set_index("class_index")["class_name"].sort_index()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for row, (name, loader) in enumerate((("train (augmented)", train_loader), ("val", val_loader))):
    images, labels, _ = next(iter(loader))
    for col in range(8):
        ax = axes[row, col]
        ax.imshow((images[col] * std + mean).clamp(0, 1).permute(1, 2, 0))
        ax.set_title(f"{name}\n{class_names[labels[col].item()]}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

All checks PASS → the shared pipeline is ready for Gate 3 (E1 SimpleCNN), provided `03_duplicate_check.ipynb` has been run and its decision recorded in DECISION_LOG.